# 제출용 추론 노트북

**이 노트북을 동결(Save Version)하여 제출합니다.**

이 노트북은 **학습된 모델을 불러와 예측만** 수행합니다.

---

## 지켜야 할 규칙

**1. 날짜를 하드코딩하지 마세요.** 예측 기간은 반드시 `PRED_DATES`를 참조해야 합니다.  
운영진은 `PRED_START`, `PRED_END`만 바꿔 실행합니다. 날짜가 코드에 박혀 있으면 채점이 불가능합니다.

**2. 예측 대상 시각은 매일 14:00 KST입니다.**  
설정 셀에서 `PRED_DATES`의 각 원소를 **14:00 KST가 포함된 `datetime`** 으로 만들어 줍니다.


위성 API의 `date` 파라미터는 **UTC 기준**이므로 설정 셀의 `to_api_datetime()`을 사용하세요.  
예: `2026-08-24 14:00 KST → 202608240500 (UTC)`

**3. 마지막에 `pred` DataFrame을 만드세요.** 형식은 아래 설명을 참고하세요.

## 사전 준비

1. 학습된 모델을 파일로 저장 → **캐글 Dataset으로 업로드** (형식 자유)
   - Public으로 설정하거나 운영진 계정에 공유
2. 우측 **Add Input** 으로 연결
   - 대회 데이터셋 (`station_list.csv`)
3. **Session options → Internet: On**

> `station_list.csv` 는 **어떤 지점을 채점하는지만** 알려줍니다.

## 1. 설정 — 수정하지 마세요

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  ★ 운영진 수정 구역 — 채점 시 아래 3줄만 교체합니다 ★               ║
# ╚═══════════════════════════════════════════════════════════════════╝
API_KEY    = ""                # 기상청 API Hub 인증키
PRED_START = "20260624"        # 예측 시작일 (YYYYMMDD)
PRED_END   = "20260630"        # 예측 종료일 (YYYYMMDD)
# ═══════════════════════════════════════════════════════════════════════

import os, sys, glob
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests

if not API_KEY:
    sys.exit("API_KEY를 입력하세요. (Kaggle Secrets는 사용할 수 없습니다)")

# ── 평가 기준 시각: 수정하지 마세요 ─────────────────────────────
EVAL_HOUR = 14
EVAL_MINUTE = 0

# 예측 대상 일시
# 중요: PRED_DATES의 각 원소는 '날짜만'이 아니라 매일 14:00 KST의 datetime 입니다.
_start_dt = datetime.strptime(PRED_START, "%Y%m%d").replace(
    hour=EVAL_HOUR, minute=EVAL_MINUTE, second=0, microsecond=0
)
_last_dt = datetime.strptime(PRED_END, "%Y%m%d").replace(
    hour=EVAL_HOUR, minute=EVAL_MINUTE, second=0, microsecond=0
)

if _start_dt > _last_dt:
    sys.exit("PRED_START는 PRED_END보다 늦을 수 없습니다.")

PRED_DATES = []
_dt = _start_dt
while _dt <= _last_dt:
    PRED_DATES.append(_dt)
    _dt += timedelta(days=1)

# 방어적 검증: 예측 대상 시각이 실수로 00시 등으로 바뀌는 것을 차단
if any(
    (pd.Timestamp(d).hour != EVAL_HOUR) or
    (pd.Timestamp(d).minute != EVAL_MINUTE)
    for d in PRED_DATES
):
    sys.exit("PRED_DATES 생성 오류: 모든 예측 대상 시각은 14:00 KST여야 합니다.")

def _as_kst(dt):
    """naive datetime은 KST로 해석하고, timezone-aware 값은 KST로 변환합니다."""
    ts = pd.Timestamp(dt)
    if ts.tzinfo is None:
        return ts.tz_localize("Asia/Seoul")
    return ts.tz_convert("Asia/Seoul")

# 위성 API 요청용 시각 문자열 생성 함수
def to_api_datetime(obs_dt, target_dt=None):

    if target_dt is None:
        target_dt = obs_dt

    obs = _as_kst(obs_dt)
    target = _as_kst(target_dt)

    
    if target.hour != EVAL_HOUR or target.minute != EVAL_MINUTE:
        raise ValueError(
            f"잘못된 예측 대상 시각: {target}. "
            f"target_dt는 {EVAL_HOUR:02d}:{EVAL_MINUTE:02d} KST여야 합니다."
        )

   
    if obs > target:
        raise ValueError(
            f"미래 위성영상 사용 금지: obs_dt={obs}, target_dt={target}. "
            "추론에는 예측 대상 시각까지 관측된 위성영상만 사용할 수 있습니다."
        )

    # KMA GK-2A LE1B API date는 UTC 기준
    return obs.tz_convert("UTC").strftime("%Y%m%d%H%M")


from urllib.parse import urlparse, parse_qs

API_AUDIT_LOG = []

if not hasattr(requests.sessions.Session, "_competition_original_request"):
    requests.sessions.Session._competition_original_request = requests.sessions.Session.request

_ORIGINAL_REQUEST = requests.sessions.Session._competition_original_request

def _audit_request(self, method, url, **kwargs):
    try:
        parsed = urlparse(str(url))
        host = parsed.netloc

        if "apihub.kma.go.kr" in host:
            query = parse_qs(parsed.query, keep_blank_values=True)

            params = kwargs.get("params")
            if isinstance(params, dict):
                for k, v in params.items():
                    if k == "authKey":
                        continue
                    if isinstance(v, (list, tuple)):
                        query[k] = [str(x) for x in v]
                    else:
                        query[k] = [str(v)]

            def _first(name):
                vals = query.get(name, [])
                return vals[0] if vals else None

            API_AUDIT_LOG.append({
                "method": str(method).upper(),
                "host": host,
                "path": parsed.path,
                "date": _first("date"),
                "sDate": _first("sDate"),
                "eDate": _first("eDate"),
            })
    except Exception as e:
        print(f"[경고] API 로그 기록 실패: {type(e).__name__}: {e}")

    return _ORIGINAL_REQUEST(self, method, url, **kwargs)

requests.sessions.Session.request = _audit_request

# 평가 대상 지점 ── station_list.csv 에 정의된 공식 96개
# glob 반환 순서에 의존하지 않고, STN_ID 96개 후보를 검증해서 선택합니다.
_hits = sorted(glob.glob("/kaggle/input/**/station_list.csv", recursive=True))
if not _hits:
    sys.exit("station_list.csv 를 찾을 수 없습니다. Add Input을 확인하세요.")

_valid_station_files = []
for _p in _hits:
    try:
        _tmp = pd.read_csv(_p)
        if "STN_ID" not in _tmp.columns:
            continue
        _ids = tuple(sorted(_tmp["STN_ID"].dropna().astype(int).unique().tolist()))
        if len(_ids) == 96:
            _valid_station_files.append((_p, _ids))
    except Exception:
        continue

if not _valid_station_files:
    sys.exit(
        "STN_ID 96개를 가진 station_list.csv를 찾지 못했습니다. "
        "공식 대회 데이터셋 연결을 확인하세요."
    )

_station_id_sets = {ids for _, ids in _valid_station_files}
if len(_station_id_sets) > 1:
    _paths = [p for p, _ in _valid_station_files]
    sys.exit(
        "서로 다른 96지점 station_list.csv가 여러 개 발견되었습니다. "
        "공식 대회 데이터셋만 남기거나 중복 파일명을 변경하세요.\n"
        + "\n".join(_paths)
    )

_station_path, _station_ids = _valid_station_files[0]
STATIONS = list(_station_ids)

if len(_valid_station_files) > 1:
    print(
        f"[경고] 동일한 96지점 station_list.csv가 {len(_valid_station_files)}개 발견되었습니다. "
        f"다음 파일을 사용합니다: {_station_path}"
    )
else:
    print(f"station_list.csv: {_station_path}")

print(f"예측 기간 : {PRED_START} ~ {PRED_END}  ({len(PRED_DATES)}일)")
print(f"예측 대상 시각 : 매일 {EVAL_HOUR:02d}:{EVAL_MINUTE:02d} KST")
print(f"평가 지점 : {len(STATIONS)}개")
print(
    "API 시각 예시 : "
    f"{PRED_DATES[0].strftime('%Y-%m-%d %H:%M')} KST -> "
    f"{to_api_datetime(PRED_DATES[0])} UTC"
)

## 2. 자유 구현 ★ 여기만 채우세요 ★

### `pred` 반환 형식

| 컬럼 | 내용 |
|---|---|
| `Date` | 정수 `YYYYMMDD` |
| `STN_ID` | 정수 지점번호 |
| `TA` | 기온 예측값 (°C) |
| `HM` | 습도 예측값 (%) |

`PRED_DATES` × `STATIONS` 의 **모든 조합**이 정확히 한 번씩 있어야 하고, 결측이 없어야 합니다.

### 평가/위성 시각 규칙 — 중요

- **예측 대상 시각은 매일 14:00 KST**입니다.
- 설정 셀의 `PRED_DATES`에는 이미 각 평가일 14:00 KST가 들어 있습니다.
- 위성 입력은 **해당 `target_dt`까지 관측된 자료**를 사용할 수 있습니다.
- GK-2A LE1B API의 `date`는 **UTC 기준**입니다.
- `to_api_datetime(obs_dt, target_dt)`를 사용하면 KST 기준 규칙을 확인한 뒤 UTC `YYYYMMDDHHMM`으로 변환합니다.
- **naive `obs_dt`는 KST로 해석됩니다. 이미 UTC인 값을 timezone 없이 넣지 마세요. UTC 값을 사용할 경우 timezone-aware UTC Timestamp로 전달하세요.**
- 예: `2026-08-24 14:00 KST → 202608240500 UTC`
- 시간 피처는 **학습 때 정의한 시간 기준과 동일하게** 만드세요. KST 기준 학습이면 14, UTC 기준 학습이면 5입니다.

### 유의사항

- **학습 코드를 넣지 마세요.** 추론 전용입니다. 실행 시간이 비정상적으로 길면 재학습으로 간주될 수 있습니다.
- **무작위성을 제거하세요.** 같은 입력에 항상 같은 출력이 나와야 합니다.
- **학습 때와 동일한 방식으로 피처를 만드세요.**
- 평가기간의 위성자료를 추론 입력으로 사용할 수 있지만, 이를 이용해 모델을 추가 학습·파인튜닝하면 안 됩니다.
- API 오류를 `except Exception:`으로 숨기지 말고, 디버깅할 때는 예외 종류와 메시지를 확인하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  ↓↓↓ 자유 구현 영역: 추론만 수행합니다 ↓↓↓
# ═══════════════════════════════════════════════════════════════════

import importlib.util
import subprocess

# Kaggle에는 torch가 기본 설치되어 있습니다. 나머지는 없을 때만 설치합니다.
for _module, _package in [
    ("catboost", "catboost==1.2.8"),
    ("xarray", "xarray"),
    ("h5netcdf", "h5netcdf"),
]:
    if importlib.util.find_spec(_module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _package])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch가 필요합니다. Kaggle 기본 GPU/CPU 이미지를 사용하세요.")

# 학습 산출물 Dataset 폴더를 엄격히 한 개만 선택합니다.
_predictor_hits = sorted(
    glob.glob("/kaggle/input/**/submission_predictor.py", recursive=True)
)
_artifact_candidates = []
_required_model_files = {
    "feature_spec.json",
    "ensemble_component_weights.csv",
    "lstm_model.pt",
    "lstm_normalization.npz",
    "catboost_TA.cbm",
    "catboost_HM.cbm",
}
for _predictor_path in _predictor_hits:
    _candidate = os.path.dirname(_predictor_path)
    if all(os.path.isfile(os.path.join(_candidate, _name)) for _name in _required_model_files):
        _artifact_candidates.append(_candidate)

_artifact_candidates = sorted(set(_artifact_candidates))
if len(_artifact_candidates) != 1:
    raise RuntimeError(
        "제출 모델 Dataset 폴더를 정확히 한 개 찾을 수 있어야 합니다. "
        f"현재 후보={_artifact_candidates}"
    )

MODEL_DIR = _artifact_candidates[0]
_module_path = os.path.join(MODEL_DIR, "submission_predictor.py")
_spec = importlib.util.spec_from_file_location("submission_predictor", _module_path)
_predictor = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_predictor)

import torch

# 공식 station_list.csv의 위도·경도·고도만 사용합니다.
station_frame = pd.read_csv(_station_path)

# 2025 결측 실험(70/1,456 파일 = 4.81%)을 반영한 상한입니다.
# API가 전부 막힌 경우에는 평균값 제출로 숨기지 않고 즉시 실패합니다.
MAX_MISSING_FRACTION = 0.05
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pred = _predictor.predict_from_api(
    api_key=API_KEY,
    pred_dates=PRED_DATES,
    stations=station_frame,
    artifact_dir=MODEL_DIR,
    to_api_datetime=to_api_datetime,
    cache_dir="/kaggle/working/gk2a_cache",
    failure_csv="/kaggle/working/satellite_download_failures.csv",
    device=DEVICE,
    max_missing_fraction=MAX_MISSING_FRACTION,
)

print(f"model_dir={MODEL_DIR}")
print(f"device={DEVICE}, pred_rows={len(pred)}")
print(pred.head().to_string(index=False))


## 3. 제출 파일 생성 — 수정하지 마세요

In [ ]:
# ── PRED_DATES 표현 점검 ─────────────────────────────────────
# 참가자가 자유 구현 중 PRED_DATES를 date/UTC Timestamp 등으로 바꿔도
# 최종 제출 행 검증은 PRED_START/PRED_END에서 독립적으로 수행합니다.
# 따라서 여기서는 중단하지 않고 경고만 합니다.
_bad_times = []
for _d in PRED_DATES:
    try:
        _ts = pd.Timestamp(_d)
        if _ts.tzinfo is None:
            _kst = _ts
        else:
            _kst = _ts.tz_convert("Asia/Seoul").tz_localize(None)

        if _kst.hour != EVAL_HOUR or _kst.minute != EVAL_MINUTE:
            _bad_times.append(_d)
    except Exception:
        _bad_times.append(_d)

if _bad_times:
    print(
        "[경고] 현재 PRED_DATES에 14:00 KST가 아닌 표현이 포함되어 있습니다. "
        "최종 제출 날짜/행 수는 PRED_START/PRED_END에서 독립 검증합니다. "
        f"예시: {_bad_times[:3]}"
    )

# ── KMA API 감사 로그 저장 ───────────────────────────────────
_audit_out = "/kaggle/working/api_audit_log.csv"
if "API_AUDIT_LOG" in globals():
    _audit_df = pd.DataFrame(
        API_AUDIT_LOG,
        columns=["method", "host", "path", "date", "sDate", "eDate"],
    )
    _audit_df.to_csv(_audit_out, index=False)

    if len(_audit_df):
        _non_le1b = _audit_df[
            ~_audit_df["path"].fillna("").str.contains("/LE1B/", regex=False)
        ]
        if len(_non_le1b):
            print(
                f"[경고] /LE1B/ 이외 KMA API 경로 호출이 {len(_non_le1b)}건 기록되었습니다. "
                f"감사 로그: {_audit_out}"
            )
        else:
            print(f"API 감사 로그 저장: {_audit_out} ({len(_audit_df)}건)")
    else:
        print(f"API 감사 로그 저장: {_audit_out} (기록 0건)")

# ── pred 검증 ─────────────────────────────────────────────
if "pred" not in dir():
    sys.exit("자유 구현 영역에서 'pred' DataFrame을 만들어야 합니다.")
if not isinstance(pred, pd.DataFrame):
    sys.exit(f"pred 는 DataFrame 이어야 합니다 (현재: {type(pred).__name__})")

need = {"Date", "STN_ID", "TA", "HM"}
if not need <= set(pred.columns):
    sys.exit(f"pred 컬럼 부족: {sorted(need - set(pred.columns))}")

# 날짜 × 지점의 모든 조합이 정확히 한 번씩 있어야 합니다
# 기대 날짜 × 지점 조합은 자유 구현 영역에서 바뀔 수 있는 PRED_DATES가 아니라
# 운영진 설정값 PRED_START/PRED_END에서 독립적으로 재계산합니다.
_ref_dates = []
_ref_d = datetime.strptime(PRED_START, "%Y%m%d")
_ref_end = datetime.strptime(PRED_END, "%Y%m%d")
while _ref_d <= _ref_end:
    _ref_dates.append(_ref_d)
    _ref_d += timedelta(days=1)

_want = {
    (int(d.strftime("%Y%m%d")), s)
    for d in _ref_dates
    for s in STATIONS
}
_got  = [(int(a), int(b)) for a, b in
         zip(pred["Date"].astype(int), pred["STN_ID"].astype(int))]
if len(_got) != len(set(_got)):
    sys.exit("pred 에 중복된 (Date, STN_ID) 조합이 있습니다.")
if set(_got) != _want:
    miss, extra = _want - set(_got), set(_got) - _want
    sys.exit(f"pred 행 구성 오류 — 누락 {len(miss)}개, 불필요 {len(extra)}개\n"
             f"  누락 예시: {sorted(miss)[:3]}\n"
             f"  불필요 예시: {sorted(extra)[:3]}")

# ── 제출 파일 구성 ────────────────────────────────────────
submission = pd.DataFrame({
    "ID": pred["Date"].astype(int).astype(str) + "_"
          + pred["STN_ID"].astype(int).astype(str),
    "TA": np.clip(pd.to_numeric(pred["TA"], errors="coerce"), -50, 50).round(2),
    "HM": np.clip(pd.to_numeric(pred["HM"], errors="coerce"), 0, 100).round(2),
}).sort_values("ID").reset_index(drop=True)

if submission[["TA", "HM"]].isna().any().any():
    n = int(submission[["TA", "HM"]].isna().any(axis=1).sum())
    sys.exit(f"결측 예측값 {n}행 — 모든 행에 값이 있어야 합니다.")

out = "/kaggle/working/submission.csv"
submission.to_csv(out, index=False)

print("=" * 52)
print(f"  저장 완료: {out}")
print(f"  {len(submission)}행 = {len(STATIONS)}지점 x {len(_ref_dates)}일")
print(f"  TA {submission.TA.min():.1f} ~ {submission.TA.max():.1f} C")
print(f"  HM {submission.HM.min():.1f} ~ {submission.HM.max():.1f} %")
print("=" * 52)
print(submission.head().to_string(index=False))

## 제출 전 점검

- [ ] 자유 구현 영역에 **날짜가 하드코딩되어 있지 않은가** (`PRED_DATES` 참조 확인)
- [ ] `PRED_DATES`의 예측 대상 시각이 매일 **14:00 KST**인가
- [ ] 사용하는 모든 위성 관측시각 `obs_dt`가 해당 예측의 **`target_dt` 이하**인가
- [ ] 과거 날짜 위성영상을 사용하더라도 평가일 14:00 KST 이후의 **미래 영상은 사용하지 않는가**
- [ ] GK-2A LE1B API의 `date`가 **KST가 아니라 UTC**로 변환되어 요청되는가
- [ ] 예: **14:00 KST → 05:00 UTC (`...0500`)**
- [ ] naive `obs_dt`는 KST로 해석된다는 점을 확인했는가 (이미 UTC인 naive datetime을 넣지 않았는가)
- [ ] 시간 피처가 있다면 **학습 때 정의한 시간 기준과 동일한가** (KST 기준 학습이면 14, UTC 기준 학습이면 5)
- [ ] 자유 구현 영역에 **학습 코드가 남아 있지 않은가**
- [ ] 평가기간 위성자료로 모델을 추가 학습·파인튜닝하고 있지 않은가
- [ ] 위성 다운로드 실패를 무시한 채 전부 대체값으로 예측하고 있지는 않은가
- [ ] 모델 Dataset이 Public이거나 운영진에 공유되었는가
- [ ] API 키를 코드에 직접 입력했는가 (Secrets 미사용)
- [ ] 실행 로그에 표시된 `station_list.csv`가 공식 96지점 파일인지 확인했는가
- [ ] `/kaggle/working/api_audit_log.csv`에 예상한 KMA API 경로/시각만 기록되었는가
- [ ] 설정 셀과 제출 검증 셀의 핵심 규칙을 임의 변경하지 않았는가

### 날짜 교체 리허설

`PRED_START` / `PRED_END` 를 **다른 기간으로 바꿔** Run All 해 보세요.  
운영진 채점과 똑같은 상황입니다.

예측 대상 시각은 매일 **14:00 KST**여야 합니다.  
위성 API 요청 시각은 사용하는 `obs_dt`에 따라 달라지며, `to_api_datetime(obs_dt, target_dt)`가 **UTC**로 변환합니다.

예를 들어:

```text
2026-08-24 14:00 KST -> API date 202608240500 UTC
2026-08-24 13:00 KST -> API date 202608240400 UTC
2026-08-23 23:00 KST -> API date 202608231400 UTC
```

모든 경우 `obs_dt <= target_dt`를 만족해야 합니다.

### 재현성 확인

Copy & Edit 로 사본을 만들어 그대로 Run All 했을 때 `submission.csv` 가  
**동일한 값**으로 나오는지 확인하세요. 달라진다면 무작위성이 남아 있는 것입니다.

### 동결 및 제출

1. **Save Version → "Save & Run All (Commit)"** ("Quick Save"는 동결로 인정되지 않습니다)
2. **Versions** 탭에서 버전 번호·저장 시각 확인 (UTC 표시, KST = UTC + 9h)
3. **Share** 에서 운영진 계정을 Collaborator로 추가